#  System Health Check

Before running complex routing or training tasks, use this notebook to verify that your environment, configuration, and database connections are healthy.

**Checks Performed:**
1.  **Python Environment**: Imports, versions, and paths.
2.  **Configuration**: Validating `artemis.yaml`.
3.  **Database**: Connecting to Postgres and checking table stats.
4.  **Router Checkpoints**: Verifying model files exist.

---

In [11]:
%load_ext autoreload
%autoreload 2

In [8]:
# 1. Environment Check
import sys
import os
from pathlib import Path
import torch

# Setup Project Root
current_dir = Path(os.getcwd())
# Assuming we are in examples/ops/
project_root = current_dir.parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f" Python: {sys.version.split()[0]}")
print(f" Torch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f" Project Root: {project_root}")

try:
    import artemis_final
    print(" Artemis Package: OK")
except ImportError as e:
    print(f" Artemis Package: FAILED ({e})")
    sys.exit(1)

✅ Python: 3.12.12
✅ Torch: 2.9.1 (CUDA: False)
✅ Project Root: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router
✅ Artemis Package: OK


In [9]:
# 2. Configuration Validation
from artemis_final.common.config_loader import load_global_config

try:
    config = load_global_config()
    print(" Config Load: OK")
    print(f"   - DB URL: {config.db.url.split('@')[1] if '@' in config.db.url else '***'}")
    print(f"   - Router Checkpoint: {config.router.checkpoint_path}")
    print(f"   - Device: {config.router.device}")
except Exception as e:
    print(f" Config Load: FAILED ({e})")

✅ Config Load: OK
   - DB URL: localhost:5432/vlmrouter
   - Router Checkpoint: checkpoints/best_multitask_router_v1.pt
   - Device: cpu


In [10]:
# 3. Database Connection Check
import pandas as pd
from sqlalchemy import create_engine, text

if 'config' in locals():
    try:
        engine = create_engine(config.db.url)
        with engine.connect() as conn:
            # Simple query to check connection
            res = conn.execute(text("SELECT 1")).fetchone()
            print(" DB Connection: OK")
            
            # Check tables (optional)
            # inspector = inspect(engine)
            # tables = inspector.get_table_names()
            # print(f"   - Tables found: {len(tables)}")
            
    except Exception as e:
        print(f" DB Connection: FAILED ({e})")
else:
    print("️ Skipping DB check (config failed)")

✅ DB Connection: OK


In [14]:
# 4. Checkpoint Verification
if 'config' in locals():
    ckpt_path = Path(project_root) / "artemis_final" / config.router.checkpoint_path
    if ckpt_path.exists():
        print(f" Router Checkpoint found: {ckpt_path.name}")
        stat = ckpt_path.stat()
        print(f"   - Size: {stat.st_size / (1024*1024):.2f} MB")
    else:
        # Check common fallback locations
        alternatives = [
            project_root / "artemis_final/checkpoints/old_checkpoints/best_pairwise_router.pt",
            project_root / "artemis_final/checkpoints/best_multitask_router_v1.pt"
        ]
        found = False
        for alt in alternatives:
            if alt.exists():
                print(f"️ Main checkpoint missing, but found alternative: {alt.name}")
                found = True
                break
        if not found:
            print(f" Router Checkpoint: MISSING (Checked {ckpt_path} and alternatives)")
else:
    print("️ Skipping Checkpoint check")

✅ Router Checkpoint found: best_multitask_router_v1.pt
   - Size: 254.90 MB
